# 🚀 Serving RAG with FastAPI

## From Notebook to API

A RAG system in a notebook is great for learning.
A RAG system behind an API is something you can actually deploy and share.

```
Before (notebook):
  ask("What is RAG?")  →  prints answer

After (API):
  POST /ask  {"question": "What is RAG?"}  →  {"answer": "..."}
```

Anyone (a frontend, mobile app, Slack bot) can call your API.

In [1]:
# !pip install fastapi uvicorn sentence-transformers

In [2]:
# Here's the complete FastAPI app — save this as rag_api.py and run it
# We'll walk through each part below

app_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- Setup ---
app = FastAPI(title="My RAG API")
model = SentenceTransformer("all-MiniLM-L6-v2")

# Knowledge base (in production: load from a database or vector store)
documents = [
    "RAG combines retrieval with LLM generation.",
    "Embeddings convert text into vectors for similarity search.",
    "Chunking splits long documents into smaller pieces.",
]
doc_embeddings = model.encode(documents)

# --- Request/Response models ---
class Question(BaseModel):
    question: str
    top_k: int = 2  # How many docs to retrieve (optional, default 2)

class Answer(BaseModel):
    answer: str
    sources: list[str]

# --- Endpoints ---
@app.get("/")
def health_check():
    return {"status": "ok", "docs": len(documents)}

@app.post("/ask", response_model=Answer)
def ask(body: Question):
    # 1. Embed the question
    q_emb = model.encode(body.question)

    # 2. Find relevant docs
    scores = cosine_similarity([q_emb], doc_embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:body.top_k]
    sources = [documents[i] for i in top_idx]

    # 3. (Would call LLM here with sources as context)
    answer = f"Based on the context: {sources[0]}"

    return Answer(answer=answer, sources=sources)
'''

print("FastAPI app code:")
print(app_code)

FastAPI app code:

from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- Setup ---
app = FastAPI(title="My RAG API")
model = SentenceTransformer("all-MiniLM-L6-v2")

# Knowledge base (in production: load from a database or vector store)
documents = [
    "RAG combines retrieval with LLM generation.",
    "Embeddings convert text into vectors for similarity search.",
    "Chunking splits long documents into smaller pieces.",
]
doc_embeddings = model.encode(documents)

# --- Request/Response models ---
class Question(BaseModel):
    question: str
    top_k: int = 2  # How many docs to retrieve (optional, default 2)

class Answer(BaseModel):
    answer: str
    sources: list[str]

# --- Endpoints ---
@app.get("/")
def health_check():
    return {"status": "ok", "docs": len(documents)}

@app.post("/ask", response_model=Answer)
def ask(body: Question):


In [3]:
# Save the app to a file
with open("rag_api.py", "w") as f:
    f.write(app_code.strip())

print("Saved to rag_api.py")
print()
print("To run it:")
print("  uvicorn rag_api:app --reload")
print()
print("Then test it:")
print('  curl -X POST http://localhost:8000/ask \\')
print('       -H "Content-Type: application/json" \\')
print('       -d \'{"question": "What is RAG?"}\'') 

Saved to rag_api.py

To run it:
  uvicorn rag_api:app --reload

Then test it:
  curl -X POST http://localhost:8000/ask \
       -H "Content-Type: application/json" \
       -d '{"question": "What is RAG?"}'


In [4]:
# Test the same logic inline (without starting a server)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
documents = [
    "RAG combines retrieval with LLM generation.",
    "Embeddings convert text into vectors for similarity search.",
    "Chunking splits long documents into smaller pieces.",
]
doc_embeddings = model.encode(documents)

# Simulate the /ask endpoint
question = "What is RAG?"
q_emb = model.encode(question)
scores = cosine_similarity([q_emb], doc_embeddings)[0]
top_idx = np.argsort(scores)[::-1][:2]
sources = [documents[i] for i in top_idx]
answer = f"Based on the context: {sources[0]}"

print("Simulated API response:")
print({"answer": answer, "sources": sources})

Simulated API response:
{'answer': 'Based on the context: RAG combines retrieval with LLM generation.', 'sources': ['RAG combines retrieval with LLM generation.', 'Chunking splits long documents into smaller pieces.']}


## API Endpoints Summary

```
GET  /         → health check, shows how many docs are indexed
POST /ask      → {"question": "..."}  →  {"answer": "...", "sources": [...]}
```

FastAPI auto-generates interactive docs at:  
**http://localhost:8000/docs** ← open this in your browser after running!

---
Next: `01_Production_Checklist.ipynb`